In [ ]:
# Додавання бібліотек
import math
import pprint
import random
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

# Можливі значення змінних
tss_range = (17, 27)
ta_range = (6, 17)
ph_range = (2.8, 4.0)
grape_kinds = ["blue", "green"]

# Кількість екземплярів
samples = 14
generated_data = {
    # Вид винограду
    "kind": [random.choice(grape_kinds) for _ in range(samples)],
    # Точна кислотність
    "ph": [round(random.uniform(*ph_range), 1) for _ in range(samples)],
    # Відносна кислотність
    "ta": [random.randint(*ta_range) for _ in range(samples)],
    # Загальна кількість розчинних твердих речовин
    "tss": [random.randint(*tss_range) for _ in range(samples)],
}
kind_labels = {value: number for number, value in enumerate(grape_kinds)}
print(kind_labels)


In [ ]:
df = pd.DataFrame(data=generated_data)
print(df)

Евклідова норма:

$$
\displaystyle
new=\frac{old}{sum}
$$

$$
\displaystyle
sum=\sqrt{\sum_{n=1}^{N}{feature_n^2}}
$$

- $N$: кількість екземплярів
- $new$: нормалізований екземпляр
- $old$: оригінальний екземпляр

In [ ]:
for sample_number, sample in df.iterrows():
    df.at[sample_number, "kind"] = kind_labels[df["kind"][sample_number]]

feature_sums = dict()
for feature_name, feature_samples in df.items():
    if feature_name == "kind":
        continue

    df[feature_name] = df[feature_name].astype(float)
    samples_sum = math.sqrt(sum(sample**2 for sample in feature_samples))
    for sample_number, sample in enumerate(feature_samples):
        normalized_sample = sample / samples_sum
        df.at[sample_number, feature_name] = round(normalized_sample, 2)

    feature_sums[feature_name] = samples_sum
print(df)

Середнє значення:

$$
center=\frac{1}{N}\sum_{n=1}^{N}feature_n
$$

In [ ]:
class_centers = dict()
for class_number in kind_labels.values():
    center = {
        "ph": float(round(df[df["kind"] == class_number]["ph"].mean(), 2)),
        "ta": float(round(df[df["kind"] == class_number]["ta"].mean(), 2)),
        "tss": float(round(df[df["kind"] == class_number]["tss"].mean(), 2)),
    }
    class_centers[class_number] = center
pprint.pprint(class_centers)

In [ ]:
new_sample = {
    "ph": round(random.uniform(*ph_range), 1),
    "ta": random.randint(*ta_range),
    "tss": random.randint(*tss_range),
}
print(new_sample)

In [ ]:
for feature, value in new_sample.items():
    feature_sum = feature_sums[feature]
    new_sample[feature] = round(value / feature_sum, 2)
print(new_sample)

$$
distance=\sqrt{\sum_{j=1}^{J}{new_j^2-center_j^2}}
$$

- $J$: кількість ознак

In [ ]:
distances = dict()
for class_number in class_centers.keys():
    center_values = np.array(list(class_centers[class_number].values()))
    new_sample_values = np.array(list(new_sample.values()))
    distance_to_class = math.sqrt(sum((new_sample_values - center_values) ** 2))
    distances[distance_to_class] = class_number
print(distances)

In [ ]:
sorted_distances = sorted(list(distances.keys()))
print(sorted_distances)

In [ ]:
predicted_class = 0
if len(sorted_distances) == 1 or sorted_distances[0] != sorted_distances[1]:
    predicted_class = distances[sorted_distances[0]]
elif sorted_distances[0] == sorted_distances[1]:
    number_of_samples_one = len(df[df["kind"] == distances[sorted_distances[0]]])
    number_of_samples_two = len(df[df["kind"] == distances[sorted_distances[1]]])
    predicted_class = (
        distances[sorted_distances[0]]
        if number_of_samples_one >= number_of_samples_two
        else distances[sorted_distances[1]]
    )
print(predicted_class)

In [ ]:
df.loc[-1] = [predicted_class] + list(new_sample.values())
df.index = df.index + 1
df = df.sort_index()
df["kind"] = df["kind"].astype(int)
print(df)